# NASA C-MAPSS FD001 - Dataset Inspection

**Project:** Axlero Project 2 - Machinery Predictive Maintenance

## 1. Objective
Inspect the raw FD001 files before any modelling: check the files and schema, missing values, duplicates, engine cycle ranges, constant sensors and target-label availability, then write the data dictionary and the inspection notes.

This notebook only **reads** the raw files. It does not create labels, engineer features or train models.

**To run:** pick the project's `.venv` as the kernel, then **Run All**. If the data files are missing, it stops in section 3 with instructions.

## 2. Dataset source
- Official page: https://data.nasa.gov/dataset/cmapss-jet-engine-simulated-data
- File needed from `CMAPSSData.zip`: `train_FD001.txt`, `test_FD001.txt`, `RUL_FD001.txt` -> put them in `data/raw/CMAPSSData/`
- Reference: Saxena, Goebel, Simon, Eklund (2008), *Damage Propagation Modeling for Aircraft Engine Run-to-Failure Simulation*, PHM08.

In [1]:
import sys
from datetime import date
from pathlib import Path

import pandas as pd


def find_project_root():
    """Go up from the current folder until the project root (the one containing src/) is found."""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "src" / "data" / "load_data.py").exists():
            return folder
    raise FileNotFoundError("Project root not found. Open the 'fd001-dataset-inspection' folder in VS Code.")


PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.load_data import COLUMN_NAMES, RAW_DIR, REPORTS_DIR, check_files, load_fd001
from src.data.inspect_dataset import (
    check_rul_count, count_duplicates, count_infinite, dataframe_to_markdown,
    engine_cycle_summary, find_constant_columns, make_data_dictionary, missing_value_table,
)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
print("Project root:", PROJECT_ROOT)

Project root: C:\Users\xa03a\Downloads\fd001-dataset-inspection-simple\fd001-dataset-inspection


## 3. File validation
Are the three required files present and non-empty?

In [2]:
status = check_files(RAW_DIR)
print(status)
if set(status.values()) != {"ok"}:
    raise FileNotFoundError(
        "Some files are missing or empty. Download CMAPSSData.zip from the NASA page and copy "
        "train_FD001.txt, test_FD001.txt and RUL_FD001.txt into data/raw/CMAPSSData/, then run again."
    )

{'train_FD001.txt': 'ok', 'test_FD001.txt': 'ok', 'RUL_FD001.txt': 'ok'}


## 4. Data loading
The loader checks that there are exactly 26 columns, then assigns the column names. Nothing is dropped or changed.

In [3]:
train, test, rul = load_fd001(RAW_DIR)
print("train:", train.shape, "| test:", test.shape, "| RUL values:", len(rul))

train: (20631, 26) | test: (13096, 26) | RUL values: 100


## 5. Dataset overview

In [4]:
print("Column names:", list(train.columns))
print("Engines in train:", train["unit_number"].nunique(), "| engines in test:", test["unit_number"].nunique())
train.head()

Column names: ['unit_number', 'time_cycles', 'op_setting_1', 'op_setting_2', 'op_setting_3', 'sensor_1', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_5', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_10', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21']
Engines in train: 100 | engines in test: 100


,unit_number,time_cycles,op_setting_1,op_setting_2,op_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,sensor_6,sensor_7,sensor_8,sensor_9,sensor_10,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,21.61,554.36,2388.06,9046.19,1.3,47.47,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,21.61,553.75,2388.04,9044.07,1.3,47.49,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,21.61,554.26,2388.08,9052.94,1.3,47.27,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,21.61,554.45,2388.11,9049.48,1.3,47.13,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,21.61,554.00,2388.06,9055.15,1.3,47.28,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044


In [5]:
test.head()

,unit_number,time_cycles,op_setting_1,op_setting_2,op_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,sensor_6,sensor_7,sensor_8,sensor_9,sensor_10,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
0,1,1,0.0023,0.0003,100.0,518.67,643.02,1585.29,1398.21,14.62,21.61,553.90,2388.04,9050.17,1.3,47.20,521.72,2388.03,8125.55,8.4052,0.03,392,2388,100.0,38.86,23.3735
1,1,2,-0.0027,-0.0003,100.0,518.67,641.71,1588.45,1395.42,14.62,21.61,554.85,2388.01,9054.42,1.3,47.50,522.16,2388.06,8139.62,8.3803,0.03,393,2388,100.0,39.02,23.3916
2,1,3,0.0003,0.0001,100.0,518.67,642.46,1586.94,1401.34,14.62,21.61,554.11,2388.05,9056.96,1.3,47.50,521.97,2388.03,8130.10,8.4441,0.03,393,2388,100.0,39.08,23.4166
3,1,4,0.0042,0.0000,100.0,518.67,642.44,1584.12,1406.42,14.62,21.61,554.07,2388.03,9045.29,1.3,47.28,521.38,2388.05,8132.90,8.3917,0.03,391,2388,100.0,39.00,23.3737
4,1,5,0.0014,0.0000,100.0,518.67,642.51,1587.19,1401.92,14.62,21.61,554.16,2388.01,9044.55,1.3,47.31,522.15,2388.03,8129.54,8.4031,0.03,390,2388,100.0,38.99,23.4130


## 6. Data types

In [6]:
pd.DataFrame({"train": train.dtypes.astype(str), "test": test.dtypes.astype(str)})

,train,test
unit_number,int64,int64
time_cycles,int64,int64
op_setting_1,float64,float64
op_setting_2,float64,float64
op_setting_3,float64,float64
sensor_1,float64,float64
sensor_2,float64,float64
sensor_3,float64,float64
sensor_4,float64,float64
sensor_5,float64,float64


## 7. Missing-value analysis

In [7]:
train_missing = missing_value_table(train)
test_missing = missing_value_table(test)
print("Missing cells - train:", int(train_missing["missing_count"].sum()), "| test:", int(test_missing["missing_count"].sum()))
train_missing

Missing cells - train: 0 | test: 0


,missing_count,missing_percentage
unit_number,0,0.0
time_cycles,0,0.0
op_setting_1,0,0.0
op_setting_2,0,0.0
op_setting_3,0,0.0
sensor_1,0,0.0
sensor_2,0,0.0
sensor_3,0,0.0
sensor_4,0,0.0
sensor_5,0,0.0


## 8. Duplicate checks and basic quality checks

In [8]:
train_dups, test_dups = count_duplicates(train), count_duplicates(test)
print("Duplicates (train):", train_dups)
print("Duplicates (test) :", test_dups)
print("Infinite values (train / test):", count_infinite(train), "/", count_infinite(test))

train_constant, test_constant = find_constant_columns(train), find_constant_columns(test)
print("Constant channels (train):", train_constant)
print("Constant channels (test) :", test_constant)

Duplicates (train): {'full_row_duplicates': 0, 'unit_time_duplicates': 0}
Duplicates (test) : {'full_row_duplicates': 0, 'unit_time_duplicates': 0}
Infinite values (train / test): 0 / 0
Constant channels (train): ['op_setting_3', 'sensor_1', 'sensor_5', 'sensor_10', 'sensor_16', 'sensor_18', 'sensor_19']
Constant channels (test) : ['op_setting_3', 'sensor_1', 'sensor_5', 'sensor_10', 'sensor_16', 'sensor_18', 'sensor_19']


## 9. Engine-cycle summaries
First cycle, last cycle and number of rows for every engine.

In [9]:
train_engines = engine_cycle_summary(train)
test_engines = engine_cycle_summary(test)
print("Rows per engine - train:", train_engines["n_rows"].min(), "to", train_engines["n_rows"].max(),
      "| test:", test_engines["n_rows"].min(), "to", test_engines["n_rows"].max())
print("Engines with non-consecutive cycles - train:", int((~train_engines["consecutive"]).sum()),
      "| test:", int((~test_engines["consecutive"]).sum()))
print("Engines not starting at cycle 1 - train:", int((train_engines["first_cycle"] != 1).sum()),
      "| test:", int((test_engines["first_cycle"] != 1).sum()))
train_engines.head()

Rows per engine - train: 128 to 362 | test: 31 to 303
Engines with non-consecutive cycles - train: 0 | test: 0
Engines not starting at cycle 1 - train: 0 | test: 0


,first_cycle,last_cycle,n_rows,consecutive
unit_number,,,,
1,1,192,192,True
2,1,287,287,True
3,1,179,179,True
4,1,189,189,True
5,1,269,269,True


## 10. Descriptive statistics
Operational settings and sensors (training file).

In [10]:
channels = [c for c in COLUMN_NAMES if c.startswith(("op_setting_", "sensor_"))]
train_stats = train[channels].describe().T
train_stats

,count,mean,std,min,25%,50%,75%,max
op_setting_1,20631.0,-0.000009,2.187313e-03,-0.0087,-0.0015,0.0000,0.0015,0.0087
op_setting_2,20631.0,0.000002,2.930621e-04,-0.0006,-0.0002,0.0000,0.0003,0.0006
op_setting_3,20631.0,100.000000,0.000000e+00,100.0000,100.0000,100.0000,100.0000,100.0000
sensor_1,20631.0,518.670000,0.000000e+00,518.6700,518.6700,518.6700,518.6700,518.6700
sensor_2,20631.0,642.680934,5.000533e-01,641.2100,642.3250,642.6400,643.0000,644.5300
sensor_3,20631.0,1590.523119,6.131150e+00,1571.0400,1586.2600,1590.1000,1594.3800,1616.9100
sensor_4,20631.0,1408.933782,9.000605e+00,1382.2500,1402.3600,1408.0400,1414.5550,1441.4900
sensor_5,20631.0,14.620000,5.329200e-15,14.6200,14.6200,14.6200,14.6200,14.6200
sensor_6,20631.0,21.609803,1.388985e-03,21.6000,21.6100,21.6100,21.6100,21.6100
sensor_7,20631.0,553.367711,8.850923e-01,549.8500,552.8100,553.4400,554.0100,556.0600


## 11. Target-label availability
- The documented schema has **no RUL column** in train or test (26 columns: unit, cycle, 3 settings, sensors).
- Training engines run until failure; test engines stop *before* failure.
- `RUL_FD001.txt` holds the true RUL of each **test** engine: cycles remaining after its last observed cycle. By convention, line *i* belongs to test engine *i* (NASA's page does not spell this out).
- Training RUL can generally be derived as `max(time_cycles of that engine) - time_cycles`. **This is not done here.**

In [11]:
has_rul_column = any("rul" in c.lower() for c in train.columns) or any("rul" in c.lower() for c in test.columns)
print("RUL column in train/test files:", has_rul_column)

rul_check = check_rul_count(test, rul)
print("RUL file vs test engines:", rul_check)
print("RUL values: min", rul["rul"].min(), "| max", rul["rul"].max(), "| mean", round(rul["rul"].mean(), 2))
print("Test unit numbers are exactly 1..N:", sorted(test["unit_number"].unique()) == list(range(1, rul_check["test_engines"] + 1)))

# Preview only (not a label table): last observed cycle of the first test engines next to the RUL file value
preview = test_engines[["last_cycle"]].head(5).copy()
preview["rul_file_value"] = rul["rul"].head(5).to_numpy()
preview

RUL column in train/test files: False
RUL file vs test engines: {'test_engines': 100, 'rul_values': 100, 'match': True}
RUL values: min 7 | max 145 | mean 75.52
Test unit numbers are exactly 1..N: True


,last_cycle,rul_file_value
unit_number,,
1,31,112
2,49,98
3,126,69
4,106,82
5,98,91


## 12. Data dictionary generation
Observed fields (dtype, missing, unique counts) come from the **training** file.
Sensor and operational-setting units are `Not specified/verified`: NASA's documentation gives no meanings or units, and the Saxena et al. (2008) paper does not say which `sensor_N` column matches which table row. The paper's candidate names appear only in `notes`, marked UNCONFIRMED.

In [12]:
dictionary = make_data_dictionary(train)
REPORTS_DIR.mkdir(exist_ok=True)
dictionary.to_csv(REPORTS_DIR / "data_dictionary.csv", index=False, encoding="utf-8-sig")
print("Saved reports/data_dictionary.csv")
dictionary.drop(columns=["notes"])

Saved reports/data_dictionary.csv


,column_name,category,description,observed_dtype,unit,missing_count,missing_percentage,unique_count
0,unit_number,identifier,Engine (unit) identifier,int64,Not applicable (identifier),0,0.0,100
1,time_cycles,time index,Time in operational cycles,int64,cycles,0,0.0,362
2,op_setting_1,operational setting,Operational setting 1. Meaning: Not specified/...,float64,Not specified/verified,0,0.0,158
3,op_setting_2,operational setting,Operational setting 2. Meaning: Not specified/...,float64,Not specified/verified,0,0.0,13
4,op_setting_3,operational setting,Operational setting 3. Meaning: Not specified/...,float64,Not specified/verified,0,0.0,1
5,sensor_1,sensor measurement,Sensor measurement 1. Meaning: Not specified/v...,float64,Not specified/verified,0,0.0,1
6,sensor_2,sensor measurement,Sensor measurement 2. Meaning: Not specified/v...,float64,Not specified/verified,0,0.0,310
7,sensor_3,sensor measurement,Sensor measurement 3. Meaning: Not specified/v...,float64,Not specified/verified,0,0.0,3012
8,sensor_4,sensor measurement,Sensor measurement 4. Meaning: Not specified/v...,float64,Not specified/verified,0,0.0,4051
9,sensor_5,sensor measurement,Sensor measurement 5. Meaning: Not specified/v...,float64,Not specified/verified,0,0.0,1


## 13. Inspection report generation
Writes `reports/dataset_inspection.md` using the numbers computed above.

In [13]:
tr_rows = train_engines["n_rows"]
te_rows = test_engines["n_rows"]
dtype_table = pd.DataFrame({"train_dtype": train.dtypes.astype(str), "test_dtype": test.dtypes.astype(str)})
stats_table = train_stats[["count", "mean", "std", "min", "50%", "max"]]
file_sizes = pd.DataFrame({"size_bytes": [(RAW_DIR / n).stat().st_size for n in status]}, index=list(status))

report = f"""# NASA C-MAPSS FD001 - Dataset Inspection Notes

> Generated by `notebooks/FD001_Dataset_Inspection.ipynb` from the real files on {date.today()}. Re-run the notebook to refresh.

## 1. Project objective
Axlero Project 2 (Machinery Predictive Maintenance) uses NASA C-MAPSS FD001. This task inspects the raw data before modelling. Out of scope: label creation, feature engineering, model training, streaming inference.

## 2. Dataset source and download instructions
- Official page: https://data.nasa.gov/dataset/cmapss-jet-engine-simulated-data (resource `CMAPSSData.zip`)
- Download `CMAPSSData.zip`, then copy `train_FD001.txt`, `test_FD001.txt`, `RUL_FD001.txt` into `data/raw/CMAPSSData/`.
- NASA lists the license as "License not specified", so the raw files are not committed to Git.
- Reference: Saxena, Goebel, Simon, Eklund (2008), Damage Propagation Modeling for Aircraft Engine Run-to-Failure Simulation, PHM08.

## 3. Dataset files
Training engines run until failure; test engines stop before failure; `RUL_FD001.txt` has the true RUL of the test engines.

{dataframe_to_markdown(file_sizes)}

## 4. Dataset dimensions and engine counts
| file | rows | columns | engines | rows per engine (min / mean / max) |
| --- | --- | --- | --- | --- |
| train_FD001 | {len(train):,} | {train.shape[1]} | {train["unit_number"].nunique()} | {tr_rows.min()} / {tr_rows.mean():.2f} / {tr_rows.max()} |
| test_FD001 | {len(test):,} | {test.shape[1]} | {test["unit_number"].nunique()} | {te_rows.min()} / {te_rows.mean():.2f} / {te_rows.max()} |

RUL file: {rul_check["rul_values"]} values; test engines: {rul_check["test_engines"]}; match: {rul_check["match"]}. NASA documents FD001 as 100 train and 100 test trajectories, one condition, one fault mode (HPC degradation).

## 5. Column schema and data types
26 columns, no header row: unit number, cycle, 3 operational settings, 21 sensors.

{dataframe_to_markdown(dtype_table)}

## 6. Missing-value analysis
Missing cells - train: {int(train_missing["missing_count"].sum())}; test: {int(test_missing["missing_count"].sum())}. Infinite values - train: {count_infinite(train)}; test: {count_infinite(test)}. Nothing was removed or filled.

## 7. Duplicate analysis
| file | duplicated full rows | duplicated (unit_number, time_cycles) pairs |
| --- | --- | --- |
| train_FD001 | {train_dups["full_row_duplicates"]} | {train_dups["unit_time_duplicates"]} |
| test_FD001 | {test_dups["full_row_duplicates"]} | {test_dups["unit_time_duplicates"]} |

## 8. Basic sensor statistics
Training file, operational settings and sensors.

{dataframe_to_markdown(stats_table)}

Constant channels - train: {", ".join(train_constant) or "none"}. Test: {", ".join(test_constant) or "none"}.

## 9. Target-label availability
- Explicit RUL column in train/test files: {has_rul_column}. The documented schema has none.
- `RUL_FD001.txt` = true RUL of each test engine (cycles remaining after its last observed cycle). By convention line i belongs to test engine i; NASA's page does not state this.
- Training RUL can generally be derived as `max(time_cycles of the engine) - time_cycles` (not done in this task).
- RUL file values: min {rul["rul"].min()}, max {rul["rul"].max()}, mean {rul["rul"].mean():.2f}. Test unit numbers are exactly 1..N: {sorted(test["unit_number"].unique()) == list(range(1, rul_check["test_engines"] + 1))}.

## 10. Units and documentation limitations
- Verified: `time_cycles` is in cycles; RUL is in cycles.
- NASA's documentation names sensors only as "sensor measurement 1..N" and gives no meanings or units. The Saxena et al. (2008) paper (Table 2) lists symbols and units for 21 sensors but does not say which `sensor_N` column is which, so every sensor/operational-setting unit is recorded as `Not specified/verified`. The paper's candidate names are shown in the data dictionary notes, marked UNCONFIRMED.
- Check `data/raw/CMAPSSData/readme.txt` (if present) or another authoritative source before interpreting channels physically.

## 11. Initial observations
- Train: {len(train):,} rows from {train["unit_number"].nunique()} engines. Test: {len(test):,} rows from {test["unit_number"].nunique()} engines.
- Missing values: {int(train_missing["missing_count"].sum()) + int(test_missing["missing_count"].sum())} in total. Duplicate rows/keys: {sum(train_dups.values()) + sum(test_dups.values())} in total.
- Engines with non-consecutive cycles: {int((~train_engines["consecutive"]).sum()) + int((~test_engines["consecutive"]).sum())}. Engines not starting at cycle 1: {int((train_engines["first_cycle"] != 1).sum()) + int((test_engines["first_cycle"] != 1).sum())}.
- Constant channels in train: {len(train_constant)} ({", ".join(train_constant) or "none"}).

## 12. Known limitations
- Checks are basic: schema, missing values, duplicates, cycle ranges, constant channels.
- Outliers, noise, train/test differences and sensor correlations are not analysed.
- Sensor meanings/units are unverified (section 10).
- The pairing of RUL lines to test engines is the usual convention, not stated by NASA.

## 13. Instructions to reproduce
Put the three files in `data/raw/CMAPSSData/`, open `notebooks/FD001_Dataset_Inspection.ipynb`, select the `.venv` kernel and Run All. See `README.md`.
"""
(REPORTS_DIR / "dataset_inspection.md").write_text(report, encoding="utf-8")
print("Saved reports/dataset_inspection.md")

Saved reports/dataset_inspection.md


## 14. Final checklist
An unticked box is a finding to review, not necessarily an error.

In [14]:
checks = {
    "All 3 files present and non-empty": set(status.values()) == {"ok"},
    "Train and test have 26 columns": train.shape[1] == 26 and test.shape[1] == 26,
    "100 engines in train and in test (as NASA documents)": train["unit_number"].nunique() == 100 and test["unit_number"].nunique() == 100,
    "No missing values": train_missing["missing_count"].sum() == 0 and test_missing["missing_count"].sum() == 0,
    "No duplicates": sum(train_dups.values()) + sum(test_dups.values()) == 0,
    "All engines start at cycle 1 with consecutive cycles": bool(train_engines["consecutive"].all() and test_engines["consecutive"].all()
                                                                 and (train_engines["first_cycle"] == 1).all() and (test_engines["first_cycle"] == 1).all()),
    "RUL file length equals number of test engines": rul_check["match"],
    "No RUL column in train/test": not has_rul_column,
    "Data dictionary written": (REPORTS_DIR / "data_dictionary.csv").exists(),
    "Inspection report written": (REPORTS_DIR / "dataset_inspection.md").exists(),
}
for text, ok in checks.items():
    print(f"[{'x' if ok else ' '}] {text}")

[x] All 3 files present and non-empty
[x] Train and test have 26 columns
[x] 100 engines in train and in test (as NASA documents)
[x] No missing values
[x] No duplicates
[x] All engines start at cycle 1 with consecutive cycles
[x] RUL file length equals number of test engines
[x] No RUL column in train/test
[x] Data dictionary written
[x] Inspection report written
